In [0]:
#===============================================================================
#  Notebook 8: Public NCDOT Traffic API
#
#
#===============================================================================


import sys


# ── Configuration ────────────────────────────────────────────────────────────
# When run by a job: parameters arrive via sys.argv
# When run interactively: dbutils widgets provide a UI with dev defaults
if len(sys.argv) >= 3 and sys.argv[1].startswith("/"):
    BASE_PATH = sys.argv[1]
    CATALOG   = sys.argv[2]
else:
    dbutils.widgets.text("base_path", "/Volumes/main/default/dot_lakehouse")
    dbutils.widgets.text("catalog", "main")
    BASE_PATH = dbutils.widgets.get("base_path")
    CATALOG   = dbutils.widgets.get("catalog")

# ── Notebook: NCDOT Live Traffic API Ingest ───────────────────────────────
# No API key needed. All endpoints are unauthenticated and public.

import requests
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.appName("NCDOT_LiveFeed").getOrCreate()

BASE_URL = "https://eapps.ncdot.gov/services/traffic-prod/v1"

# ── 1. Pull all active incidents statewide ────────────────────────────────
resp = requests.get(f"{BASE_URL}/incidents?active=true&verbose=true", timeout=15)
incident_data = resp.json()
print(f"Active incidents: {incident_data.get('activeIncidentCount', 0)}")

# ── 2. Pull all counties (use IDs to query per-county later) ─────────────
counties  = requests.get(f"{BASE_URL}/counties", timeout=15).json()
roads     = requests.get(f"{BASE_URL}/roads",    timeout=15).json()

# ── 3. Pull road-specific incidents for your corridors ────────────────────
target_roads = ["I-40", "I-85", "I-77", "I-95", "US-74"]
road_lookup  = {r["name"]: r["id"] for r in roads}

road_incidents = {}
for road in target_roads:
    road_id = road_lookup.get(road)
    if road_id:
        r = requests.get(
            f"{BASE_URL}/roads/{road_id}/incidents?verbose=true&recent=true",
            timeout=15
        )
        road_incidents[road] = r.json()

# ── 4. Flatten to Spark DataFrame and write to Delta ──────────────────────
# Road-specific endpoints return full incident objects;
# the statewide endpoint only returns IDs.
rows = []
seen_ids = set()
for road, incidents in road_incidents.items():
    for incident in incidents:
        iid = incident.get("id")
        if iid in seen_ids:
            continue
        seen_ids.add(iid)
        rows.append({
            "incident_id":      iid,
            "road_name":        incident.get("road"),
            "description":      incident.get("reason"),
            "incident_type":    incident.get("incidentType"),
            "severity":         incident.get("severity"),
            "latitude":         incident.get("latitude"),
            "longitude":        incident.get("longitude"),
            "start_time":       incident.get("start"),
            "lanes_affected":   incident.get("condition"),
            "direction":        incident.get("direction"),
            "county":           incident.get("countyName"),
            "is_active":        True,
            "ingested_at":      datetime.utcnow().isoformat(),
        })

print(f"Corridor incidents collected: {len(rows)}")

if rows:
    df = spark.createDataFrame(rows)
    df.write.format("delta").mode("append") \
      .save(f"{BASE_PATH}/streaming/ncdot_live_incidents")
    display(df)

In [0]:
# ── Notebook: NCDOT Camera Snapshot Ingestion ─────────────────────────────
# Uses the NCDOT traffic-prod API to discover real camera IDs and image URLs,
# then fetches and stores snapshot images.

import requests, base64, time
from datetime import datetime, timezone
from pyspark.sql import functions as F

# ── Real camera IDs from the NCDOT traffic-prod/v1/cameras API ────────────
# Discovered via: GET /cameras → list of IDs, then GET /cameras/{id} → detail
# 330 cameras on target corridors; 2 per road selected here.

KNOWN_CAMERAS = [
    # (camera_id, label, route, lat, lon, image_url)
    (5,   "I-40 Exit 270 - US 15-501",             "I-40",  35.950774, -78.998909,
     "https://eapps.ncdot.gov/services/traffic-prod/v1/cameras/images?filename=I40_US15-501.jpg"),
    (7,   "I-40 Exit 274 - NC 751",                "I-40",  35.904138, -78.960028,
     "https://eapps.ncdot.gov/services/traffic-prod/v1/cameras/images?filename=I40_NC751.jpg"),
    (207, "I-85 - Mallard Creek Church Rd",         "I-85",  35.332142, -80.744362,
     "https://cfms.services.ncdot.gov/snapshots/chan-5170_h.jpg"),
    (317, "I-85 @ Concord Mills Blvd",              "I-85",  35.368685, -80.714536,
     "https://eapps.ncdot.gov/services/traffic-prod/v1/cameras/images?filename=0I0085S0049-25.JPG"),
    (382, "US 74/NC 133 at Isabel Holmes Bridge",   "US-74", 34.252679, -77.946729,
     "https://eapps.ncdot.gov/services/traffic-prod/v1/cameras/images?filename=MLK_HolmesBr.JPG"),
    (402, "US 74 (Eastwood Rd) at Racine Dr",       "US-74", 34.244393, -77.862922,
     "https://eapps.ncdot.gov/services/traffic-prod/v1/cameras/images?filename=OldEastwood_Racine.JPG"),
    (806, "I-95 NB @ NC 87",                        "I-95",  34.97674,  -78.85554,
     "https://cfes.services.ncdot.gov/snapshots/chan-5772_h.jpg"),
    (808, "I-95 @ NC 24",                           "I-95",  35.04254,  -78.80431,
     "https://cfes.services.ncdot.gov/snapshots/chan-5773_h.jpg"),
    (90,  "I-77 - N of Clanton Rd",                 "I-77",  35.19843,  -80.882463,
     "https://eapps.ncdot.gov/services/traffic-prod/v1/cameras/images?filename=I77_ClantonRd.jpg"),
    (91,  "I-77 - S of Clanton Rd",                 "I-77",  35.1915,   -80.88591,
     "https://cfms.services.ncdot.gov/snapshots/chan-5137_h.jpg"),
]

def fetch_camera_snapshot(image_url: str) -> dict:
    """Fetch a camera JPEG and return metadata + base64 image."""
    try:
        resp = requests.get(image_url, timeout=10)
        if resp.status_code == 200 and resp.headers.get("content-type","").startswith("image"):
            return {
                "success":      True,
                "status_code":  resp.status_code,
                "image_bytes":  len(resp.content),
                "image_b64":    base64.b64encode(resp.content).decode("utf-8"),
                "content_type": resp.headers.get("content-type"),
                "last_modified":resp.headers.get("last-modified"),
            }
        else:
            return {"success": False, "status_code": resp.status_code}
    except Exception as e:
        return {"success": False, "error": str(e)}

# ── Poll all cameras and build a record ───────────────────────────────────
records = []
for (cam_id, label, route, lat, lon, img_url) in KNOWN_CAMERAS:
    result   = fetch_camera_snapshot(img_url)
    snapshot = {
        "camera_id":      str(cam_id),
        "label":          label,
        "route":          route,
        "latitude":       lat,
        "longitude":      lon,
        "polled_at":      datetime.now(timezone.utc).isoformat(),
        "fetch_success":  result.get("success", False),
        "image_bytes":    result.get("image_bytes", 0),
        "last_modified":  result.get("last_modified"),
        "image_b64":      result.get("image_b64"),
    }
    records.append(snapshot)
    print(f"  {'✓' if result.get('success') else '✗'} {label}: "
          f"{result.get('image_bytes', 0):,} bytes")
    time.sleep(0.5)   # be a polite client

# ── Write snapshot metadata to Delta (without the image bytes) ────────────
# Overwrite once to fix schema from earlier test run, then switch to append.
import pandas as pd
pdf = pd.DataFrame(records).drop(columns=["image_b64"])
df  = spark.createDataFrame(pdf)
df.write.format("delta").mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save(f"{BASE_PATH}/streaming/ncdot_camera_poll_log")

# ── Display latest snapshots inline in the notebook ───────────────────────
html_parts = ["<div style='display:flex;flex-wrap:wrap;gap:12px;'>"]
for rec in records:
    if rec["fetch_success"] and rec["image_b64"]:
        img_html = f"""
        <div style='border:1px solid #333;border-radius:6px;overflow:hidden;width:320px'>
          <img src='data:image/jpeg;base64,{rec["image_b64"]}'
               style='width:100%;display:block'>
          <div style='padding:6px 8px;font-size:11px;font-family:monospace;
                      background:#111;color:#ccc'>
            {rec['label']}<br>
            {rec['route']} · {rec['polled_at'][:19]}
          </div>
        </div>"""
        html_parts.append(img_html)
html_parts.append("</div>")
displayHTML("".join(html_parts))

In [0]:
# ── Camera Discovery: find new cameras on target corridors ────────────────
# Pulls all cameras from the NCDOT API, filters to target corridors,
# tests each for a valid image, and writes a full registry to Delta.

import requests, time
from datetime import datetime, timezone

BASE_URL = "https://eapps.ncdot.gov/services/traffic-prod/v1"

# Road IDs for target corridors
roads = requests.get(f"{BASE_URL}/roads", timeout=15).json()
target_roads = {"I-40": None, "I-85": None, "I-77": None, "I-95": None, "US-74": None}
for r in roads:
    if r["name"] in target_roads:
        target_roads[r["name"]] = r["id"]
road_id_to_name = {v: k for k, v in target_roads.items() if v}
print(f"Target road IDs: {target_roads}")

# Get all camera IDs then fetch details for corridor cameras
all_cameras = requests.get(f"{BASE_URL}/cameras", timeout=15).json()
print(f"Total cameras in NC: {len(all_cameras)}")

known_ids = {c[0] for c in KNOWN_CAMERAS}
corridor_cams = []
new_cams = []

for cam in all_cameras:
    detail = requests.get(f"{BASE_URL}/cameras/{cam['id']}", timeout=10).json()
    if detail.get("roadId") not in road_id_to_name:
        continue
    if detail.get("status") != "OK":
        continue

    # Test image availability
    img_url = detail.get("imageURL", "")
    img_ok = False
    img_bytes = 0
    if img_url:
        try:
            img_resp = requests.get(img_url, timeout=8)
            if img_resp.status_code == 200 and len(img_resp.content) > 500:
                img_ok = True
                img_bytes = len(img_resp.content)
        except:
            pass

    record = {
        "camera_id":     detail["id"],
        "location_name": detail.get("locationName", ""),
        "road":          road_id_to_name[detail["roadId"]],
        "mile_marker":   detail.get("mileMarker"),
        "latitude":      detail["latitude"],
        "longitude":     detail["longitude"],
        "image_url":     img_url,
        "image_ok":      img_ok,
        "image_bytes":   img_bytes,
        "is_dot_camera": detail.get("isDOTCamera", False),
        "discovered_at": datetime.now(timezone.utc).isoformat(),
    }
    corridor_cams.append(record)

    if detail["id"] not in known_ids and img_ok:
        new_cams.append(record)

    time.sleep(0.05)

print(f"\nCorridor cameras (status=OK): {len(corridor_cams)}")
print(f"  With valid images: {sum(1 for c in corridor_cams if c['image_ok'])}")
print(f"  New (not in KNOWN_CAMERAS): {len(new_cams)}")

if new_cams:
    print(f"\n── New cameras with working images ──")
    for c in new_cams[:20]:
        print(f"  Camera {c['camera_id']:>4d}: {c['location_name'][:50]:50s}  "
              f"{c['road']:6s}  {c['image_bytes']:>7,} bytes")
    if len(new_cams) > 20:
        print(f"  ... and {len(new_cams) - 20} more")

# Write full camera registry to Delta
import pandas as pd
df_registry = spark.createDataFrame(pd.DataFrame(corridor_cams))
df_registry.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{BASE_PATH}/streaming/ncdot_camera_registry")
print(f"\n✓ Camera registry written to {BASE_PATH}/streaming/ncdot_camera_registry")
display(df_registry)

In [0]:
%pip install folium -q

In [0]:
# ── Live Incident Map with Camera Overlay ──────────────────────────────────

import requests, time
import folium
from folium.plugins import MarkerCluster

BASE_URL = "https://eapps.ncdot.gov/services/traffic-prod/v1"

SEVERITY_COLORS = {1: "green", 2: "orange", 3: "red", 4: "darkred"}
INCIDENT_ICONS = {
    "Construction":            "wrench",
    "Night Time Construction": "wrench",
    "Maintenance":             "cog",
    "Disabled Vehicle":        "car",
    "Reported Incident":       "exclamation-triangle",
    "Accident":                "exclamation-circle",
}

# ── 1. Fetch live incidents for all 5 corridors ──────────────────────────
roads_api = requests.get(f"{BASE_URL}/roads", timeout=15).json()
corridor_names = ["I-40", "I-85", "I-77", "I-95", "US-74"]
road_lookup = {r["name"]: r["id"] for r in roads_api if r["name"] in corridor_names}

incidents = []
seen = set()
for road_name, road_id in road_lookup.items():
    try:
        resp = requests.get(
            f"{BASE_URL}/roads/{road_id}/incidents?verbose=true&recent=true",
            timeout=15,
        )
        for inc in resp.json():
            iid = inc.get("id")
            if iid in seen:
                continue
            seen.add(iid)
            incidents.append(inc)
    except Exception:
        pass

print(f"Live incidents across 5 corridors: {len(incidents)}")

# ── 2. Load camera registry from Delta ────────────────────────────────────
cameras = (
    spark.read.format("delta")
    .load(f"{BASE_PATH}/streaming/ncdot_camera_registry")
    .filter("image_ok = true")
    .select("camera_id", "location_name", "road", "latitude", "longitude")
    .collect()
)
print(f"Camera locations loaded: {len(cameras)}")

# ── 3. Build Folium map ───────────────────────────────────────────────────
m = folium.Map(location=[35.55, -79.80], zoom_start=7, tiles="CartoDB dark_matter")

# Incident markers (clustered)
inc_cluster = MarkerCluster(name="Incidents").add_to(m)
for inc in incidents:
    lat, lon = inc.get("latitude"), inc.get("longitude")
    if not lat or not lon:
        continue

    sev = inc.get("severity", 1)
    color = SEVERITY_COLORS.get(sev, "blue")
    icon_name = INCIDENT_ICONS.get(inc.get("incidentType", ""), "info-sign")

    popup_html = (
        f"<div style='width:280px;font-family:sans-serif;font-size:12px'>"
        f"<b>{inc.get('road', '').strip()}</b> — {inc.get('incidentType', '')}<br>"
        f"<b>Condition:</b> {inc.get('condition', '')}<br>"
        f"<b>Location:</b> {inc.get('location', '')}<br>"
        f"<b>County:</b> {inc.get('countyName', '')}, {inc.get('city', '')}<br>"
        f"<b>Direction:</b> {inc.get('direction', '')}<br>"
        f"<b>Severity:</b> {sev}<br>"
        f"<hr style='margin:4px 0'>"
        f"<i>{inc.get('reason', '')[:200]}</i><br>"
        f"<small>Start: {inc.get('start', '')[:16]}</small>"
        f"</div>"
    )

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{inc.get('road', '').strip()} — {inc.get('condition', '')}",
        icon=folium.Icon(color=color, icon=icon_name, prefix="fa"),
    ).add_to(inc_cluster)

# Camera markers (cyan circles)
cam_group = folium.FeatureGroup(name="Cameras").add_to(m)
for cam in cameras:
    folium.CircleMarker(
        location=[cam["latitude"], cam["longitude"]],
        radius=5,
        color="cyan",
        fill=True,
        fill_opacity=0.7,
        tooltip=f"\U0001f4f7 {cam['location_name']}",
        popup=f"Camera {cam['camera_id']}: {cam['location_name']} ({cam['road']})",
    ).add_to(cam_group)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
            background:#222;padding:10px 14px;border-radius:6px;
            font-size:12px;color:#ccc;font-family:sans-serif;
            box-shadow:0 2px 6px rgba(0,0,0,0.5);">
    <b>Severity</b><br>
    <span style="color:#72b026">●</span> Minor&nbsp;
    <span style="color:#f0a30a">●</span> Moderate&nbsp;
    <span style="color:#cb2b3e">●</span> Major&nbsp;
    <span style="color:#8b0000">●</span> Severe<br>
    <span style="color:cyan">●</span> Camera
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)

# ── 4. Display inline ─────────────────────────────────────────────────────
print(f"\nMap ready: {len(incidents)} incidents + {len(cameras)} cameras")
displayHTML(m._repr_html_())